In [ ]:
!!!!!pip install langchain langchain-openai langgraph python-dotenv langchain-community duckduckgo-search

### Set up Groq API Key

To use the Groq API, you'll need an API key. If you don't already have one, create one on the [Groq Console](https://console.groq.com/keys).

In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `GROQ_API_KEY`.

In [37]:
import os
print(os.getcwd())

c:\Users\Administrator\EY-Training-Repo\eytraining\Day-15


In [38]:
import os

print(".env exists:", os.path.exists(".env"))

.env exists: True


In [36]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if GROQ_API_KEY:
    print("Key loaded:", GROQ_API_KEY[:10] + "...")
else:
    print("GROQ_API_KEY not found!")

Key loaded: None...


In [ ]:
!pip install langgraph
!pip install langchain
!pip install langchain-core
!pip install langchain-groq
!pip install python-dotenv

In [ ]:
import os, json
from dotenv import load_dotenv
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq


load_dotenv()
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.1-8b-instant",
    groq_api_key=GROQ_API_KEY
)

# --- shared state schema ---
class AgentState(TypedDict):
    goal:       str
    tasks:      List[str]
    results:    List[str]
    critique:   str
    approved:   bool
    iterations: int

In [ ]:
def planner(state: AgentState) -> AgentState:
    system = """You are a planning agent. Break the user's goal into
at most 5 concrete, actionable tasks. Respond ONLY with a
valid JSON array of strings. No preamble, no markdown."""

    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Goal: {state['goal']}")
    ]
    response = llm.invoke(messages).content.strip()

    try:
        clean = response.replace("```json","").replace("```","").strip()
        tasks = json.loads(clean)
    except json.JSONDecodeError:
        tasks = [response]   # fallback: treat whole response as one task

    print(f"\n[Planner] Generated {len(tasks)} tasks:")
    for i, t in enumerate(tasks): print(f"  {i+1}. {t}")

    return {**state, "tasks": tasks}

In [ ]:
#! pip install langchain langchain-openai langgraph python-dotenv langchain-community duckduckgo-search
# Imports and Environment (OPEN_AI key setup)
!pip install langchain_groq
import os, json
from dotenv import load_dotenv
from typing import TypedDict, List
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, END

In [ ]:
# %%capture
!pip install langchain langchain_groq langchain_community python-dotenv

In [ ]:
!pip install -U ddgs

In [ ]:
search = DuckDuckGoSearchRun()
def executor(state:AgentState) -> AgentState:
  results = []
  critique_ctx = ""
  if state["critique"]:
    critique_ctx = f"\n\nYour previous attempt was rejected. Previous critique: {state['critique']}"

  for task in state["tasks"]:
    system=f"""You are an execution agent. Complete the task thoroughly. Use web search if you need
    current information. {critique_ctx}"""

    # try web search for research tasks
    search_ctx = ""

    try:
      search_result = search.run(task[:100])
      search_ctx = f"\n\nWeb search result for context: \n{search_result[:800]}"
    except:
      pass

    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Task: {task}{search_ctx}")
    ]

    result = llm.invoke(messages).content

    results.append(result)
    print(f"\n[Executor] Task: {task[:60]}...\n  Result: {result}")
  return {**state, "results": results, "iterations": state["iterations"] + 1}

In [ ]:
# Agent with LLM-as-a-judge
def verifier(state: AgentState) -> AgentState:
    # safety net — approve after 3 iterations regardless
    if state["iterations"] >= 3:
        print("[Verifier] Max iterations reached — force approving.")
        return {**state, "approved": True}

    combined_results = "\n\n".join(
        f"Task {i+1}: {t}\nResult: {r}"
        for i, (t, r) in enumerate(zip(state["tasks"], state["results"]))
    )
    system = """You are a quality verifier. Evaluate the results against the
original goal using this rubric:
- Completeness: Does it fully address the goal? (0-0.4)
- Accuracy:     Is the information correct and specific? (0-0.3)
- Clarity:      Is it well-structured and clear? (0-0.3)
- Latency:      Does it take a reasonable amount of time to complete? (0-0.3)
- Tokens:       Number of tokens used (0-10000)
Sum the scores for a total between 0.0 and 1.0.
Respond ONLY as JSON: {"score":0.9, "completeness_score": 0.35, "accuracy_score": 0.2, "clarity_score":0.15, "approved": true, "critique": "...", latenc}"""

    messages = [
        SystemMessage(content=system),
        HumanMessage(content=f"Original goal: {state['goal']}\n\nResults:\n{combined_results}")
    ]
    raw = llm.invoke(messages).content.strip()
    try:
        clean = raw.replace("```json","").replace("```","").strip()
        verdict = json.loads(clean)
        completeness  = verdict.get("completeness_score", False)
        accuracy      = verdict.get("accuracy_score", False)
        clarity       = verdict.get("clarity_score", False)
        approved      = verdict.get("approved", False)
        critique  = verdict.get("critique", "")
        score     = verdict.get("score", 0)
    except:
        approved, critique, score = False, raw, 0

    print(f"\n[Verifier] Score: {score:.2f} | Approved: {approved}")
    if not approved: print(f"  Critique: {critique}")
    return {**state, "approved": approved, "critique": critique}

def route_after_verify(state: AgentState) -> str:
    return "end" if state["approved"] else "executor"

In [ ]:
# --- run it ---
######Main######
graph = StateGraph(AgentState)

graph.add_node("planner",  planner)
graph.add_node("executor", executor)
graph.add_node("verifier", verifier)


graph.add_edge("planner", "executor")
graph.add_edge("executor", "verifier")
graph.add_conditional_edges(
    "verifier",
    route_after_verify,
    {"end": END, "executor": "executor"}
)

graph.set_entry_point("planner")


app = graph.compile()
initial_state: AgentState = {
    "goal":       "Research and summarise the top 3 trends in agriculture for 2025",
    "tasks":      [],
    "results":    [],
    "critique":   "",
    "approved":   False,
    "iterations": 0
}

final_state = app.invoke(initial_state)